# TextThreat Colab Training Notebook

This notebook trains the thesis models for **TextThreat — AI-Powered Detection of Digital Well-Being Risks with Cybersecurity Analytics**.

It trains:

1. SVM + TF-IDF baseline for Jigsaw multi-label toxicity classification.
2. DistilBERT + LoRA improved classifier for Jigsaw multi-label toxicity classification.
3. DistilBERT + LoRA binary stress classifier for Dreaddit.

It also generates the supporting thesis evidence artifacts: sample events, classification metrics, latency, output-level privacy perturbation, fairness demo/audit, and synthetic co-occurrence results.

**Important:** run this notebook in Colab with a GPU runtime for DistilBERT training.

## 1. Select GPU Runtime

In Colab, go to:

`Runtime -> Change runtime type -> Hardware accelerator -> GPU`

The SVM baseline can run on CPU, but DistilBERT training is much faster with GPU.

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. DistilBERT training will be slow.')

## 2. Clone The Repository

This clones the GitHub repository into the Colab machine. If you forked or moved the repo, change `REPO_URL`.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/abdulmuksith3/textthreat-poc.git'
BRANCH = 'next-phase'
# Keep this import close to REPO_DIR so the cell works even when run by itself.
from pathlib import Path
REPO_DIR = Path('/content/textthreat-poc')

if not REPO_DIR.exists():
    !git clone --branch $BRANCH --single-branch $REPO_URL $REPO_DIR
else:
    print('Repository already exists:', REPO_DIR)

os.chdir(REPO_DIR)
!git fetch origin $BRANCH
!git checkout $BRANCH
!git pull origin $BRANCH
print('Working directory:', Path.cwd())

## 3. Install Dependencies

This installs the repository requirements, including Transformers, PEFT/LoRA, Fairlearn, Opacus, MLflow, and Splunk/OpenSearch clients.

If Colab asks to restart the runtime after installation, restart and run the notebook again from this point.

In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt

# Colab currently preinstalls an old torchao build in some runtimes.
# PEFT/LoRA does not require torchao here, and the old version makes LoRA injection fail.
!pip uninstall -y torchao


## 4. Download Datasets From Shared Google Drive Folder

The CSVs are stored in a public Google Drive folder shared with anyone who has the link. This cell downloads the folder into Colab and copies the dataset files into the repo paths required by the training scripts.

Shared folder:

```text
https://drive.google.com/drive/folders/1I2tIX7Fz4BHa_6mjn8QVBYpBTghBVbbY?usp=sharing
```

Expected local repo paths after the copy step:

```text
data/jigsaw/train.csv
data/dreaddit/dreaddit-train.csv
data/dreaddit/dreaddit-test.csv
```

Raw datasets are used locally in Colab only and should not be committed to Git.


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

PUBLIC_DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1I2tIX7Fz4BHa_6mjn8QVBYpBTghBVbbY?usp=sharing'
DOWNLOAD_DIR = Path('/content/textthreat_drive_data')

try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gdown'])
    import gdown

if not DOWNLOAD_DIR.exists() or not any(DOWNLOAD_DIR.iterdir()):
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    print('Downloading shared Drive folder...')
    gdown.download_folder(
        url=PUBLIC_DRIVE_FOLDER_URL,
        output=str(DOWNLOAD_DIR),
        quiet=False,
        use_cookies=False,
    )
else:
    print('Using existing downloaded folder:', DOWNLOAD_DIR)

csv_files = sorted(DOWNLOAD_DIR.rglob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'No CSV files found under {DOWNLOAD_DIR}. Check the public Drive link permissions.')

print('CSV files found:')
for csv_path in csv_files:
    print(' -', csv_path)

def choose_csv(label, scorer):
    ranked = sorted(((scorer(path), path) for path in csv_files), reverse=True)
    score, path = ranked[0]
    if score <= 0:
        raise FileNotFoundError(f'Could not identify {label} CSV in downloaded folder. Files: {csv_files}')
    print(f'Selected {label}: {path}')
    return path

def jigsaw_train_score(path):
    name = path.name.lower()
    full = str(path).lower()
    if name == 'train.csv' and 'dreaddit' not in full:
        return 100
    if 'jigsaw' in full and 'train' in name:
        return 90
    if 'toxic' in full and 'train' in name:
        return 80
    return 0

def dreaddit_train_score(path):
    name = path.name.lower()
    full = str(path).lower()
    if name == 'dreaddit-train.csv':
        return 100
    if 'dreaddit' in full and 'train' in name:
        return 90
    return 0

def dreaddit_test_score(path):
    name = path.name.lower()
    full = str(path).lower()
    if name == 'dreaddit-test.csv':
        return 100
    if 'dreaddit' in full and 'test' in name:
        return 90
    return 0

DATASET_COPY_MAP = {
    choose_csv('Jigsaw train.csv', jigsaw_train_score): Path('data/jigsaw/train.csv'),
    choose_csv('Dreaddit train CSV', dreaddit_train_score): Path('data/dreaddit/dreaddit-train.csv'),
    choose_csv('Dreaddit test CSV', dreaddit_test_score): Path('data/dreaddit/dreaddit-test.csv'),
}

for source, target in DATASET_COPY_MAP.items():
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
    print(f'Copied {source} -> {target} ({target.stat().st_size / 1024 / 1024:.2f} MB)')


## 5. Confirm Dataset Files

This checks that the expected files exist before training starts.

In [ ]:
expected_files = [
    Path('data/jigsaw/train.csv'),
    Path('data/dreaddit/dreaddit-train.csv'),
    Path('data/dreaddit/dreaddit-test.csv'),
]

for path in expected_files:
    print(path, 'exists:', path.exists(), 'size_mb:', round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else 'missing')

## 6. Training Configuration

Use `QUICK_TEST = True` first to verify the end-to-end training workflow quickly.

For final thesis evidence, set:

```python
QUICK_TEST = False
JIGSAW_EPOCHS = 1  # or more if you have time/GPU budget
DREADDIT_EPOCHS = 1
```

LoRA is enabled by default because the thesis describes efficient DistilBERT adaptation.

In [ ]:
import shutil
import subprocess
from pathlib import Path

QUICK_TEST = True

# Set these to True/False if you want to run only part of the training pipeline.
RUN_SVM = True
RUN_JIGSAW_DISTILBERT = True
RUN_DREADDIT_DISTILBERT = True

# LoRA trains small adapter layers instead of the full transformer, reducing memory/time.
USE_LORA = True

# One epoch is enough for a thesis PoC run; increase only if you have time/GPU budget.
JIGSAW_EPOCHS = 1
DREADDIT_EPOCHS = 1

# QUICK_TEST uses small subsets so you can verify that everything works.
SVM_SAMPLE_SIZE = 2000 if QUICK_TEST else None
JIGSAW_SAMPLE_SIZE = 1000 if QUICK_TEST else None
DREADDIT_SAMPLE_SIZE = 1000 if QUICK_TEST else None

def run_checked(command):
    print('Running:', command)
    subprocess.run(command, shell=True, check=True)

def remove_path(path):
    path = Path(path)
    if path.is_dir():
        shutil.rmtree(path)
        print('Removed old directory:', path)
    elif path.exists():
        path.unlink()
        print('Removed old file:', path)

print('QUICK_TEST:', QUICK_TEST)
print('USE_LORA:', USE_LORA)


## 7. Smoke Test Before Heavy Training

This verifies the schema, sample exports, metrics, latency, DP, fairness, co-occurrence, and SOAR-lite demo path before spending GPU time.

In [ ]:
!python scripts/smoke_test.py

## 8. Train SVM + TF-IDF Baseline

This is the thesis baseline model. It uses TF-IDF unigrams+bigrams and a calibrated one-vs-rest LinearSVC.

Output:

- `models/svm_tfidf/`
- `experiments/results/svm_metrics.json`

In [ ]:
if RUN_SVM:
    remove_path('models/svm_tfidf')
    remove_path('experiments/results/svm_metrics.json')
    cmd = 'python -m src.textthreat.train_svm --calibration-cv 3'
    if SVM_SAMPLE_SIZE is not None:
        cmd += f' --sample-size {SVM_SAMPLE_SIZE}'
    run_checked(cmd)
else:
    print('Skipping SVM baseline training.')


## 9. Train DistilBERT + LoRA On Jigsaw

This is the improved thesis classifier for six-label Jigsaw harm detection.

LoRA configuration in the repo:

- `r = 8`
- `alpha = 16`
- `dropout = 0.1`
- target modules: `q_lin`, `v_lin`

Output:

- `models/distilbert_jigsaw/`
- `experiments/results/distilbert_metrics.json`

In [ ]:
if RUN_JIGSAW_DISTILBERT:
    remove_path('models/distilbert_jigsaw')
    remove_path('experiments/results/distilbert_metrics.json')
    cmd = f'python -m src.textthreat.train_distilbert --task jigsaw --epochs {JIGSAW_EPOCHS}'
    if JIGSAW_SAMPLE_SIZE is not None:
        cmd += f' --sample-size {JIGSAW_SAMPLE_SIZE}'
    if not USE_LORA:
        cmd += ' --no-lora'
    run_checked(cmd)
else:
    print('Skipping Jigsaw DistilBERT training.')


## 10. Train DistilBERT + LoRA On Dreaddit

This trains the binary stress classifier used by the thesis to support stress signal detection and co-occurrence analytics.

Output:

- `models/distilbert_dreaddit/`
- `experiments/results/dreaddit_metrics.json`

In [ ]:
if RUN_DREADDIT_DISTILBERT:
    remove_path('models/distilbert_dreaddit')
    remove_path('experiments/results/dreaddit_metrics.json')
    cmd = f'python -m src.textthreat.train_distilbert --task dreaddit --epochs {DREADDIT_EPOCHS}'
    if DREADDIT_SAMPLE_SIZE is not None:
        cmd += f' --sample-size {DREADDIT_SAMPLE_SIZE}'
    if not USE_LORA:
        cmd += ' --no-lora'
    run_checked(cmd)
else:
    print('Skipping Dreaddit DistilBERT training.')


## 11. Generate Supporting Thesis Artifacts

These scripts create the evidence JSON files used by the thesis tables and the SIEM/SOAR demo.

The DP script is intentionally described as **output-level privacy-preserving perturbation**, not full DP-SGD training.

In [ ]:
!python -m src.textthreat.export_events --sample
!python -m src.textthreat.evaluate --sample
!python -m src.textthreat.latency --sample
!python -m src.textthreat.dp_output --sample
!python -m src.textthreat.fairness --sample
!python -m src.textthreat.cooccurrence --sample
!python soar_lite/soar_lite.py --demo

## 12. Inspect Result Files

This lists the generated artifacts. These JSON files can be copied into thesis tables or committed when they are demo/sample artifacts.

In [ ]:
from pathlib import Path
import json

for path in sorted(Path('experiments/results').glob('*')):
    print(path, round(path.stat().st_size / 1024, 2), 'KB')

print('\nDistilBERT metrics preview:')
metrics_path = Path('experiments/results/distilbert_metrics.json')
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text()) , indent=2)[:2000])
else:
    print('distilbert_metrics.json not found. Check whether training completed.')

## 14. Zip Model And Result Artifacts

This creates a single zip file you can download from Colab. It includes models and result JSONs, but not raw datasets.

## 13. Validate Model Artifact Completeness

This cell checks that trained model directories contain the files needed for inference. For LoRA, `adapter_config.json` is not enough; the folder must also contain `adapter_model.safetensors` or `adapter_model.bin`.


In [ ]:
# Fail fast if model folders are incomplete before downloading artifacts.
# For LoRA models, adapter_model.safetensors is required alongside adapter_config.json.
!python scripts/check_model_artifacts.py --root .


In [ ]:
# Re-run artifact validation here too, so an incomplete model cannot be zipped by accident.
run_checked('python scripts/check_model_artifacts.py --root .')
run_checked("zip -r textthreat_training_artifacts.zip models experiments/results data/exports/sample_textthreat_events.ndjson schema siem/splunk soar_lite README.md -x '*/__pycache__/*' '*/hf_outputs/*'")
print('Created textthreat_training_artifacts.zip')


## 15. Download Artifacts

Download the artifact zip to your machine. You can use the trained model folder later in the local or hosted demo.

In [ ]:
from google.colab import files
files.download('textthreat_training_artifacts.zip')

## 16. Optional: Upload Model To Hugging Face Hub

For the hosted demo, the cleanest deployment is to upload `models/distilbert_jigsaw/` to a Hugging Face model repository and set this environment variable in the app host:

```text
TEXTTHREAT_TOXICITY_MODEL_ID=your-username/textthreat-distilbert-jigsaw
```

The demo can also run with the instant fallback scorer, but trained model upload is better for the final thesis demo.